# Rice Leaf Disease Training Notebook

This notebook documents dataset download, training, evaluation, mobile export, and single-image inference for the PyTorch edge prototype.

## 1. Environment Check

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
print('Python:', sys.version)
print('Project root:', root)

## 2. Download the Kaggle Dataset

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download('anshulm257/rice-disease-dataset')
dataset_dir = Path(dataset_path) / 'Rice_Leaf_AUG'
print(dataset_dir)

## 3. Inspect Class Distribution

In [ ]:
from collections import Counter

from train import build_samples

samples, classes, _ = build_samples(dataset_dir)
counts = Counter(label for _, label in samples)
for idx, class_name in enumerate(classes):
    print(class_name, counts[idx])

## 4. Train the Best Model Variant

In [ ]:
%cd {root}
!python train.py --epochs 16 --batch-size 64 --image-size 128 --learning-rate 0.001 --scheduler cosine --output-dir artifacts_cosine

## 5. Export for PyTorch Mobile

In [ ]:
!python export_edge.py --checkpoint artifacts_cosine/rice_leaf_edge_model.pt --output-dir edge_artifacts_cosine

## 6. Run Single-Image Inference

In [ ]:
example_image = next(dataset_dir.rglob('*.jpg'))
print(example_image)
!python predict.py "{example_image}" --checkpoint artifacts_cosine/rice_leaf_edge_model.pt --top-k 3